# F1 Knowledge-Graph Population

Populates the three KGs (`adhoc` / `pure_sulo` / `hybrid_sulo`) from the same records of the
Kaggle *Formula 1 World Championship (1950–present)* dataset.

## Loading the Kaggle dataset

The download is pinned to dataset version 9 so that every run, and all three KGs, read identical
records (the dataset is still being updated with new races). `kagglehub` caches it under
`~/.cache/kagglehub`; it is public, so no Kaggle credentials are needed. The download also contains
`SCHEMA.md`, the author's per-column reference.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd

KAGGLE_DATASET = "atharvranjan/formula-1-world-championship-1950-present"
KAGGLE_DATASET_VERSION = 9

DATA_DIR = Path(kagglehub.dataset_download(f"{KAGGLE_DATASET}/versions/{KAGGLE_DATASET_VERSION}"))
sorted(p.name for p in DATA_DIR.iterdir())

/home/domin/miniconda3/envs/sulo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['CHANGELOG.md',
 'README.md',
 'SCHEMA.md',
 'circuits.csv',
 'constructor_results.csv',
 'constructor_standings.csv',
 'constructors.csv',
 'driver_standings.csv',
 'drivers.csv',
 'lap_times.csv',
 'pit_stops.csv',
 'practice_results.csv',
 'qualifying.csv',
 'races.csv',
 'results.csv',
 'seasons.csv',
 'sprint_results.csv',
 'status.csv',
 'tire_stints.csv',
 'weather.csv']

Missing values are written as `\N` (the Ergast convention the dataset inherits), and a few
columns (`drivers.nationality`, `qualifying.q2`, `qualifying.q3`) also use empty strings. pandas'
default NA list is switched off so only these two markers become NaN and no real string value is
silently dropped. Every column is read as a string so IDs, codes and lap times keep their exact
source form (e.g. a permanent number `44` does not become `44.0` because its column has gaps);
conversion to the schema's datatypes happens explicitly in the mapping step.

In [2]:
MISSING_VALUE_MARKERS = [r"\N", ""]

tables = {
    path.stem: pd.read_csv(path, dtype=str, keep_default_na=False, na_values=MISSING_VALUE_MARKERS)
    for path in sorted(DATA_DIR.glob("*.csv"))
}
pd.DataFrame({name: {"rows": len(df), "columns": df.shape[1]} for name, df in tables.items()}).T

,rows,columns
circuits,78,9
constructor_results,13030,5
constructor_standings,13796,7
constructors,214,5
driver_standings,35695,7
drivers,879,9
lap_times,632937,6
pit_stops,12783,7
practice_results,1901,6
qualifying,11298,9


## Building the assertions

The tables are turned once into KG-independent assertions (subject, subject class, schema
property, value). All three KGs are rendered from this one table, so they contain the same
facts by construction. The mapping from columns to schema properties lives in
`f1_population.property_sources`.

Every season (1950–2026) is in scope. Earlier seasons are sparser rather than inconsistent:
qualifying exists only from 1994 (patchy until 2003), sprints from 2021, practice only for
2025–26, and about half of the pre-2000 entries are unclassified. A missing value only omits
its triple; the entity stays. Two kinds of record are excluded:
- the **Indianapolis 500** (1950–60): it counted for the championship but was run to separate
  rules by a largely separate field (103 of its drivers never started another race);
- **shared drives** (one driver with two results in one race, up to 1978): a `RaceEntry` is
  identified by (race, driver), which is also the key session results are joined on.

The label `hasRacePoints` is `points > 0`. Its meaning depends on the era: only the top 6 or 8
scored before 2010, so before 2010 it is not the same as a top-10 finish.

In [3]:
from f1_population import EXPECTED_SET_ASIDE, NS, SCHEMA_FILES, build_assertions, read_schema, render_ntriples

OUTPUT_DIR = Path("data/f1")

entities, assertions = build_assertions(tables)
entities["class"].value_counts()

class
RaceEntry        27032
SessionResult    13782
Race              1153
Driver             761
Team               182
Circuit             78
Season              77
Position            34
Name: count, dtype: int64

## Rendering the three KGs

Each schema decides how an assertion is written: a direct edge (ad-hoc, most of Hybrid) or a
SULO path through an intermediate node `<subject>/<property>` (Pure SULO, and `hasRank` in
Hybrid). Properties a schema does not declare are set aside and must match the expected list,
so a mapping typo or schema change fails here instead of silently dropping data.

The `.nt` files are complete graphs. Leakage masking happens later by filtering
`assertions.csv` and re-rendering with `render_ntriples`, which is why the assertions and
entities are saved alongside them (reload them with `dtype=str, keep_default_na=False`).

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
entities.to_csv(OUTPUT_DIR / "entities.csv", index=False)
assertions.to_csv(OUTPUT_DIR / "assertions.csv", index=False)

summary = {}
for kg, schema_path in SCHEMA_FILES.items():
    lines, set_aside = render_ntriples(entities, assertions, read_schema(schema_path))
    assert set_aside == EXPECTED_SET_ASIDE[kg], f"{kg}: set aside {set_aside}"
    (OUTPUT_DIR / f"{kg}.nt").write_text("\n".join(lines) + "\n", encoding="utf-8")

    triples = [line[:-2].split(" ", 2) for line in lines]
    summary[kg] = {
        "triples": len(triples),
        "nodes": len({t[0] for t in triples} | {t[2] for t in triples}),
        "predicates": len({t[1] for t in triples}),
        "race entries": sum(t[1].endswith("#type>") and t[2] == f"<{NS}RaceEntry>" for t in triples),
        "points share": sum("hasRacePoints" in t[0] + t[1] and t[2].startswith('"true"') for t in triples),
    }
summary = pd.DataFrame(summary).T
summary["points share"] = (summary["points share"] / summary["race entries"]).round(3)
summary

,triples,nodes,predicates,race entries,points share
adhoc,313626,58154,30,27032,0.313
pure_sulo,701942,230214,12,27032,0.313
hybrid_sulo,359886,58193,33,27032,0.313


## Turtle samples for reading and validation

The full graphs are too large to read, so this writes a Turtle file per KG for the most recent
races only: those races plus every node they reach (seasons, circuits, race entries, session
results, drivers, teams, positions, and Pure's intermediate nodes). `precedesRace`,
`precedesSeason` and `hasTeamMember` are not followed, because they would chain to the whole graph;
their triples stay in, pointing at nodes outside the sample. Each sample is checked to be an
exact subset of its full `.nt` file, so reading the Turtle is reading the real output.
Instances get one prefix per class (`raceentry:27144`); Pure's intermediate nodes
(`.../RaceEntry/27144/hasRacePoints`) stay full IRIs, since Turtle prefixed names cannot contain `/`.

In [5]:
from rdflib import Graph

from f1_population import SULO_NS, iri, subset

N_SAMPLE_RACES = 20
SAMPLE_DIR = OUTPUT_DIR / "sample"
SAMPLE_DIR.mkdir(exist_ok=True)

sample_races = tables["races"].sort_values("date").tail(N_SAMPLE_RACES)
sample_entities, sample_assertions = subset(entities, assertions, [iri("Race", r) for r in sample_races["raceId"]])

for kg, schema_path in SCHEMA_FILES.items():
    lines, _ = render_ntriples(sample_entities, sample_assertions, read_schema(schema_path))
    full = set((OUTPUT_DIR / f"{kg}.nt").read_text(encoding="utf-8").splitlines())
    assert set(lines) <= full, f"{kg}: sample contains triples that are not in {kg}.nt"

    graph = Graph().parse(data="\n".join(lines), format="nt")
    graph.bind("", NS)
    graph.bind("sulo", SULO_NS)
    for cls in sample_entities["class"].unique():  # raceentry:27144 instead of the full IRI
        graph.bind(cls.lower(), f"{NS}{cls}/")
    graph.serialize(SAMPLE_DIR / f"{kg}.ttl", format="turtle")
    print(f"{kg}: {len(graph)} triples -> {SAMPLE_DIR / f'{kg}.ttl'}")

sample_races[["year", "round", "name", "date"]].reset_index(drop=True)

adhoc: 12963 triples -> data/f1/sample/adhoc.ttl
pure_sulo: 28580 triples -> data/f1/sample/pure_sulo.ttl
hybrid_sulo: 15375 triples -> data/f1/sample/hybrid_sulo.ttl


,year,round,name,date
0,2025,20,Mexico City Grand Prix,2025-10-26
1,2025,21,São Paulo Grand Prix,2025-11-09
2,2025,22,Las Vegas Grand Prix,2025-11-23
3,2025,23,Qatar Grand Prix,2025-11-30
4,2025,24,Abu Dhabi Grand Prix,2025-12-07
5,2026,1,Australian Grand Prix,2026-03-08
6,2026,2,Chinese Grand Prix,2026-03-15
7,2026,3,Japanese Grand Prix,2026-03-29
8,2026,4,Miami Grand Prix,2026-05-03
9,2026,5,Canadian Grand Prix,2026-05-24
